In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix as sk_confusion_matrix, 
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from catboost import CatBoostClassifier
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import logging
import time
import gc
import os
import shap
import random

In [15]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [16]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [17]:
ENSEMBLE_WEIGHTS = [0.4, 0.3, 0.3]
num_classes = 3
all_possible_labels = list(range(num_classes))

In [18]:
# Create model
def create_cnn_model_1(input_shape, num_classes):
    input_layer = Input(shape=input_shape)
    conv1 = Conv1D(filters=10, kernel_size=6, activation='relu', padding='same')(input_layer)
    conv2 = Conv1D(filters=10, kernel_size=6, activation='relu', padding='same')(conv1)
    pool = MaxPooling1D(pool_size=2, padding='same')(conv2)
    flatten = Flatten()(pool)
    dense1 = Dense(units=8, activation='relu')(flatten)
    output_layer = Dense(units=num_classes, activation='softmax', name='ecn2_output')(dense1)
    return Model(inputs=input_layer, outputs=output_layer)


def create_cnn_model_2(input_shape, num_classes):
    input_layer = Input(shape=input_shape)
    conv1 = Conv1D(filters=10, kernel_size=6, activation='relu', padding='same')(input_layer)
    conv2 = Conv1D(filters= 10, kernel_size=6, activation='relu', padding='same')(conv1)
    pool = MaxPooling1D(pool_size=2, padding='same')(conv2)
    flatten = Flatten()(pool)
    dense1 = Dense(units=8, activation='relu')(flatten)
    dropout1 = Dropout(0.5)(dense1)
    output_layer = Dense(units=num_classes, activation='softmax', name='ecn2_output')(dropout1)
    return Model(inputs=input_layer, outputs=output_layer)


def create_cnn_model_3(input_shape, num_classes):
    input_layer = Input(shape=input_shape)
    conv1 = Conv1D(filters=10, kernel_size=6, activation='relu', padding='same')(input_layer)
    conv2 = Conv1D(filters=10, kernel_size=6, activation='relu', padding='same')(conv1)
    pool = MaxPooling1D(pool_size=2, padding='same')(conv2)
    flatten = Flatten()(pool)
    output_layer = Dense(units=num_classes, activation='softmax', name='ecn3_output')(flatten)
    return Model(inputs=input_layer, outputs=output_layer)

In [19]:
#Define weights 
def weighted_ensemble_predictions(predictions, weights):
    if len(predictions) != len(weights):
        raise ValueError("Number of models must match number of weights")
    weighted_predictions = np.array([predictions[i] * weights[i] for i in range(len(predictions))])
    ensemble_predictions = np.sum(weighted_predictions, axis=0)
    ensemble_predictions = np.argmax(ensemble_predictions, axis=1)
    return ensemble_predictions

In [20]:
# Train model
def train_and_evaluate_cnn_ensemble(X_train, y_train, X_val, y_val, params):

    start_time = time.time()
    logging.info("Training CNN Ensemble...")

    X_train_np = X_train.values if isinstance(X_train, pd.DataFrame) else np.array(X_train)
    y_train_np = y_train.values if isinstance(y_train, pd.Series) else np.array(y_train)
    X_val_np = X_val.values if isinstance(X_val, pd.DataFrame) else np.array(X_val)
    y_val_np = y_val.values if isinstance(y_val, pd.Series) else np.array(y_val)

    y_train_np = y_train_np.astype(int)
    y_val_np = y_val_np.astype(int)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_np)
    X_val_scaled = scaler.transform(X_val_np)

    X_train_reshaped = np.expand_dims(X_train_scaled, axis=-1).astype(np.float32)
    X_val_reshaped = np.expand_dims(X_val_scaled, axis=-1).astype(np.float32)

    if X_train_reshaped.shape[1] == 0:
        logging.error("Error: X_train has 0 features after preprocessing/selection.")
        return None 
    input_shape = (X_train_reshaped.shape[1], 1)


    y_train_categorical = to_categorical(y_train_np, num_classes=num_classes)
    y_val_categorical = to_categorical(y_val_np, num_classes=num_classes) 

    cnn_model_1 = create_cnn_model_1(input_shape, num_classes)
    cnn_model_2 = create_cnn_model_2(input_shape, num_classes)
    cnn_model_3 = create_cnn_model_3(input_shape, num_classes)

    loss_function = 'categorical_crossentropy' 

    cnn_model_1.compile(optimizer=Adam(learning_rate=params['learning_rate']), loss=loss_function, metrics=['accuracy'])
    cnn_model_2.compile(optimizer=Adam(learning_rate=params['learning_rate']), loss=loss_function, metrics=['accuracy'])
    cnn_model_3.compile(optimizer=Adam(learning_rate=params['learning_rate']), loss=loss_function, metrics=['accuracy'])


    logging.info("Fitting CNN Models...")
    cnn_model_1.fit(X_train_reshaped, y_train_categorical, epochs=params['epochs'],
                    batch_size=params['batch_size'], verbose=0,
                    validation_data=(X_val_reshaped, y_val_categorical)) 

    cnn_model_2.fit(X_train_reshaped, y_train_categorical, epochs=params['epochs'],
                    batch_size=params['batch_size'], verbose=0,
                    validation_data=(X_val_reshaped, y_val_categorical))

    cnn_model_3.fit(X_train_reshaped, y_train_categorical, epochs=params['epochs'],
                    batch_size=params['batch_size'], verbose=0,
                    validation_data=(X_val_reshaped, y_val_categorical))


    logging.info("Predicting with CNN Models...")
    cnn_predictions_1 = cnn_model_1.predict(X_val_reshaped, batch_size=params['batch_size'], verbose=0)
    cnn_predictions_2 = cnn_model_2.predict(X_val_reshaped, batch_size=params['batch_size'], verbose=0)
    cnn_predictions_3 = cnn_model_3.predict(X_val_reshaped, batch_size=params['batch_size'], verbose=0)

    ensemble_predictions = weighted_ensemble_predictions(
        [cnn_predictions_1, cnn_predictions_2, cnn_predictions_3], ENSEMBLE_WEIGHTS)


    logging.info("Calculating Evaluation Metrics...")
    try:
        metrics = {
            'accuracy': accuracy_score(y_val_np, ensemble_predictions),
            'precision': precision_score(y_val_np, ensemble_predictions, average='macro', zero_division=0),
            'recall': recall_score(y_val_np, ensemble_predictions, average='macro', zero_division=0),
            'f1_score': f1_score(y_val_np, ensemble_predictions, average='macro', zero_division=0),
            'confusion_matrix': sk_confusion_matrix(y_val_np, ensemble_predictions,labels=all_possible_labels).tolist(),
        }
        logging.info(f"Evaluation metrics: {metrics}")
    except Exception as e:
        logging.error(f"Error calculating metrics: {e}")
        metrics = None 


    logging.info("Cleaning up models...")
    del cnn_model_1, cnn_model_2, cnn_model_3
    del cnn_predictions_1, cnn_predictions_2, cnn_predictions_3
    tf.keras.backend.clear_session() 
    gc.collect() 

    logging.info(f"CNN Ensemble trained and evaluated in {time.time() - start_time:.2f} seconds.")

    return metrics

In [21]:
#Split dataset
def load_and_preprocess_data_and_split(data_path, target_column='target'):

    logging.info(f"Loading data from: {data_path}")

    data = pd.read_csv(data_path)
    data = data.drop(columns='Unnamed: 0', errors='ignore')
    data = data.fillna(data.mean())
    print(data.shape)
    X = data.drop(target_column, axis=1)
    y = data[target_column]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    logging.info("Data loaded and preprocessed.")
    logging.info(f"Training set shape: {X_train.shape}, Validation set shape: {X_val.shape}")
    return X_train, y_train, X_val, y_val

In [22]:
def reduce_dimensionality(X, y, top_n=50):
    start_time = time.time()
    logging.info("Starting dimensionality reduction...")

    model = CatBoostClassifier(iterations=50, 
                               depth=2, 
                               learning_rate=0.1,
                               l2_leaf_reg=10,
                               loss_function='MultiClass',
                               verbose=10, 
                               random_seed=42,
                               task_type='GPU')
    model.fit(X, y, verbose=10)
    importance_df = pd.DataFrame({'feature': X.columns, 'importance': model.get_feature_importance()})
    top_features_catboost = importance_df.nlargest(top_n,'importance')['feature'].tolist()

    logging.info("Calculating SHAP values...")
    top_features_shap = None
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)

        if isinstance(shap_values, list):
            logging.info("SHAP values appear to be multiclass (list).")
            if not shap_values:
                 raise ValueError("SHAP explainer returned an empty list.")
            shap_values = [np.array(vals) for vals in shap_values]

            shapes_in_list = [vals.shape for vals in shap_values]
            logging.info(f"Shapes within SHAP values list: {shapes_in_list}")

            shap_class_importance = [np.abs(vals).mean(axis=0) for vals in shap_values]
            shap_importance = np.mean(shap_class_importance, axis=0)
        elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            logging.info(f"SHAP values: Multiclass (3D Array), Shape: {shap_values.shape}")
            shap_importance = np.abs(shap_values).mean(axis=(0, 2)) 
        else:
            if not isinstance(shap_values, np.ndarray):
                 raise TypeError(f"Expected shap_values to be list or numpy array, got {type(shap_values)}")

            logging.info(f"SHAP values appear to be binary/regression (shape: {shap_values.shape}).") # Debug log
            shap_importance = np.abs(shap_values).mean(axis=0)
  
        logging.info(f"Shape of shap_importance: {np.array(shap_importance).shape}")
        logging.info(f"Shape of X.columns: {X.columns.shape}")
  
        if np.isnan(shap_importance).any() or np.isinf(shap_importance).any():
             logging.warning("NaN or Inf detected in shap_importance values.")
             shap_importance = np.nan_to_num(shap_importance, nan=0.0, posinf=0.0, neginf=0.0)


        shap_importance_df = pd.DataFrame({'feature': X.columns,'importance': shap_importance})

        top_features_shap = shap_importance_df.nlargest(top_n, 'importance')['feature'].tolist()
        logging.info("SHAP values calculated.")
        

    except Exception as e:
        logging.error(f"Error calculating SHAP values: {e}") 
        logging.warning("Using only CatBoost features due to SHAP error.")
        top_features_shap = top_features_catboost 

    selected_features = list(set(top_features_catboost) & set(top_features_shap))

    logging.info(f"Dimensionality reduction completed in {time.time() - start_time:.2f} seconds.")
    return selected_features

In [23]:
# Optimization by Genetic Algorithm
class GeneticOptimizer:
    """Thuật toán di truyền chuẩn để tối ưu hóa siêu tham số AI"""
    def __init__(self, population_size=5, generations=3):
        self.pop_size = population_size
        self.gens = generations
        # Không gian tìm kiếm tham số (Search Space)
        self.search_space = {
            'learning_rate': [0.01, 0.001],
            'batch_size': [16, 32],
            'epochs': [45, 50, 55]
        }

    def _create_individual(self):
        """Tạo một 'cá thể' (bộ tham số) ngẫu nhiên"""
        return {k: random.choice(v) for k, v in self.search_space.items()}

    def _fitness(self, params, X_train, y_train, X_val, y_val):
        """Hàm thích nghi: Đo lường chất lượng của bộ tham số bằng F1-Score"""
        # Gọi trực tiếp hàm huấn luyện từ file của bạn
        metrics = train_and_evaluate_cnn_ensemble(X_train, y_train, X_val, y_val, params)
        return metrics['f1_score'] if metrics else 0

    def run(self, X_train, y_train, X_val, y_val):
        # Khởi tạo quần thể ban đầu
        population = [self._create_individual() for _ in range(self.pop_size)]
        
        for g in range(self.gens):
            logging.info(f"--- Thế hệ GA thứ {g+1}/{self.gens} ---")
            # Đánh giá độ thích nghi
            scores = [(self._fitness(ind, X_train, y_train, X_val, y_val), ind) for ind in population]
            scores.sort(key=lambda x: x[0], reverse=True)
            
            logging.info(f"Best F1 trong thế hệ {g+1}: {scores[0][0]}")
            
            # Chọn lọc (Selection): Lấy 2 cá thể tốt nhất
            parent1, parent2 = scores[0][1], scores[1][1]
            
            # Lai ghép (Crossover) & Đột biến (Mutation) để tạo quần thể mới
            new_population = [parent1, parent2]
            while len(new_population) < self.pop_size:
                child = {
                    'learning_rate': random.choice([parent1['learning_rate'], parent2['learning_rate']]),
                    'batch_size': random.choice([parent1['batch_size'], parent2['batch_size']]),
                    'epochs': random.choice(self.search_space['epochs']) # Mutation
                }
                new_population.append(child)
            population = new_population
            
        return scores[0][1] # Trả về bộ tham số tốt nhất sau cùng

In [24]:
#Benchmark
def run_benchmark(model_func, X_test, y_test, best_params):
        logging.info("\n=== AUTOMATION TESTING & BENCHMARKING ===")
        
        # Đo lường thời gian suy diễn (Inference Time)
        start_latency = time.time()
        # Giả lập model dự đoán để đo latency
        _ = model_func(X_test, y_test, X_test, y_test, best_params) 
        end_latency = time.time()
        
        avg_latency = (end_latency - start_latency) / len(X_test) * 1000
        logging.info(f"Độ trễ trung bình (Inference Latency): {avg_latency:.4f} ms/mẫu")
        logging.info(f"Thông lượng (Throughput): {len(X_test)/(end_latency-start_latency):.2f} mẫu/giây")

In [25]:
def main_pipeline(data_path):
    # 1. Tách tập TEST (Raw) ra ngay lập tức
    # Split 80% train_full, 20% test_raw
    X_train_full, y_train_full, X_test_raw, y_test_raw = load_and_preprocess_data_and_split(data_path)
    
    # 2. Tiếp tục tách X_train_full thành Train_sub và Val_sub để phục vụ GA
    # Bước này giúp tập X_test_raw không bị đụng chạm trong quá trình GA
    X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
    )
    
    # 3. Feature Selection (Vẫn dùng X_train_full để lấy đặc trưng tốt nhất)
    best_genes = reduce_dimensionality(X_train_full, y_train_full, top_n=50)
    
    X_train_sub_red = X_train_sub[best_genes]
    X_val_sub_red = X_val_sub[best_genes]
    X_test_raw_red = X_test_raw[best_genes] 
    
    # 4. Optimization (GA) - Truyền đủ 4 tham số (Dùng tập Sub-Train và Sub-Val)
    optimizer = GeneticOptimizer()
    best_params = optimizer.run(X_train_sub_red, y_train_sub, X_val_sub_red, y_val_sub)
    
    # 5. FINAL BENCHMARK - Chỉ dùng trên tập X_test_raw_red hoàn toàn mới
    logging.info(f"--- FINAL BENCHMARK ON RAW DATA ---")
    performance_results = run_benchmark(train_and_evaluate_cnn_ensemble, X_test_raw_red, y_test_raw, best_params)
    
    return performance_results

if __name__ == "__main__":
    data_path = "D:/Indiv project/Datasets/merged_AML_ALL.csv"
    main_pipeline(data_path)

2026-03-05 21:31:54,014 - INFO - Loading data from: D:/Indiv project/Datasets/merged_AML_ALL.csv


(3655, 22278)


2026-03-05 21:32:50,194 - INFO - Data loaded and preprocessed.
2026-03-05 21:32:50,195 - INFO - Training set shape: (2924, 22277), Validation set shape: (731, 22277)
2026-03-05 21:32:51,192 - INFO - Starting dimensionality reduction...


0:	learn: 0.9330371	total: 152ms	remaining: 7.47s
10:	learn: 0.2971473	total: 615ms	remaining: 2.18s
20:	learn: 0.1354624	total: 1.02s	remaining: 1.41s
30:	learn: 0.0750677	total: 1.43s	remaining: 879ms
40:	learn: 0.0503505	total: 1.84s	remaining: 405ms


2026-03-05 21:32:55,899 - INFO - Calculating SHAP values...


49:	learn: 0.0419505	total: 2.21s	remaining: 0us


2026-03-05 21:33:14,223 - INFO - SHAP values: Multiclass (3D Array), Shape: (2924, 22277, 3)
2026-03-05 21:33:15,617 - INFO - Shape of shap_importance: (22277,)
2026-03-05 21:33:15,618 - INFO - Shape of X.columns: (22277,)
2026-03-05 21:33:15,622 - INFO - SHAP values calculated.
2026-03-05 21:33:15,623 - INFO - Dimensionality reduction completed in 24.43 seconds.
2026-03-05 21:33:15,701 - INFO - --- Thế hệ GA thứ 1/3 ---
2026-03-05 21:33:15,702 - INFO - Training CNN Ensemble...
2026-03-05 21:33:15,964 - INFO - Fitting CNN Models...
2026-03-05 21:34:09,718 - INFO - Predicting with CNN Models...
2026-03-05 21:34:10,334 - INFO - Calculating Evaluation Metrics...
2026-03-05 21:34:10,343 - INFO - Evaluation metrics: {'accuracy': 0.9965811965811966, 'precision': 0.9984459984459985, 'recall': 0.9941002949852508, 'f1_score': 0.9962449933244325, 'confusion_matrix': [[111, 2, 0], [0, 427, 0], [0, 0, 45]]}
2026-03-05 21:34:10,344 - INFO - Cleaning up models...


2026-03-05 21:34:10,764 - WARNING - From d:\Indiv project\.venv\Lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.

2026-03-05 21:34:11,562 - INFO - CNN Ensemble trained and evaluated in 55.86 seconds.
2026-03-05 21:34:11,563 - INFO - Training CNN Ensemble...
2026-03-05 21:34:11,652 - INFO - Fitting CNN Models...
2026-03-05 21:35:01,449 - INFO - Predicting with CNN Models...
2026-03-05 21:35:02,020 - INFO - Calculating Evaluation Metrics...
2026-03-05 21:35:02,026 - INFO - Evaluation metrics: {'accuracy': 0.9965811965811966, 'precision': 0.9984459984459985, 'recall': 0.9941002949852508, 'f1_score': 0.9962449933244325, 'confusion_matrix': [[111, 2, 0], [0, 427, 0], [0, 0, 45]]}
2026-03-05 21:35:02,027 - INFO - Cleaning up models...
2026-03-05 21:35:02,743 - INFO - CNN Ensemble trained and evaluated in 51.18 seconds.
2026-03-05 21:35:02,744 - INFO - Training CNN Ensemble...
202